#Phase 3 — Feature Engineering

In [2]:
# ==========================================
# Phase 3: Feature Engineering & Preprocessing
# Customer Churn Prediction
# ==========================================

import pandas as pd
import numpy as np
import sqlite3

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

print("Libraries imported successfully!")

Libraries imported successfully!


Connect to Database

In [3]:
# Connect to SQLite database
conn = sqlite3.connect("churn.db")

print("Connected to SQLite database successfully!")

Connected to SQLite database successfully!


Load the Four Tables

In [4]:
customers = pd.read_sql_query(
    "SELECT * FROM customers",
    conn
)

accounts = pd.read_sql_query(
    "SELECT * FROM accounts",
    conn
)

services = pd.read_sql_query(
    "SELECT * FROM services",
    conn
)

churn_status = pd.read_sql_query(
    "SELECT * FROM churn_status",
    conn
)

print("Customers:", customers.shape)
print("Accounts:", accounts.shape)
print("Services:", services.shape)
print("Churn Status:", churn_status.shape)

Customers: (7043, 6)
Accounts: (7043, 6)
Services: (7043, 10)
Churn Status: (7043, 2)


Merge the Tables

In [5]:
model_df = customers.merge(
    accounts,
    on="customerID",
    how="inner"
)

model_df = model_df.merge(
    services,
    on="customerID",
    how="inner"
)

model_df = model_df.merge(
    churn_status,
    on="customerID",
    how="inner"
)

print("Model dataset shape:", model_df.shape)

Model dataset shape: (7043, 21)


Inspect the Dataset

In [6]:
display(model_df.head())

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,...,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Churn
0,7590-VHVEG,Female,0,Yes,No,1,Month-to-month,Yes,Electronic check,29.85,...,No,No phone service,DSL,No,Yes,No,No,No,No,No
1,5575-GNVDE,Male,0,No,No,34,One year,No,Mailed check,56.95,...,Yes,No,DSL,Yes,No,Yes,No,No,No,No
2,3668-QPYBK,Male,0,No,No,2,Month-to-month,Yes,Mailed check,53.85,...,Yes,No,DSL,Yes,Yes,No,No,No,No,Yes
3,7795-CFOCW,Male,0,No,No,45,One year,No,Bank transfer (automatic),42.30,...,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,No
4,9237-HQITU,Female,0,No,No,2,Month-to-month,Yes,Electronic check,70.70,...,Yes,No,Fiber optic,No,No,No,No,No,No,Yes


In [7]:
print(model_df.shape)
print(model_df.columns.tolist())

(7043, 21)
['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Churn']


Check Data Types

In [8]:
model_df.dtypes

,0
customerID,object
gender,object
SeniorCitizen,int64
Partner,object
Dependents,object
tenure,int64
Contract,object
PaperlessBilling,object
PaymentMethod,object
MonthlyCharges,float64


Convert TotalCharges

In [9]:
model_df["TotalCharges"] = pd.to_numeric(
    model_df["TotalCharges"],
    errors="coerce"
)

print(model_df["TotalCharges"].dtype)

float64


Check Missing Values After Conversion

In [10]:
missing = model_df.isnull().sum()

missing = missing[missing > 0]

display(missing)

,0
TotalCharges,11


Handle the 11 Missing TotalCharges

In [11]:
model_df["TotalCharges"] = model_df["TotalCharges"].fillna(0)

print(
    "Missing TotalCharges:",
    model_df["TotalCharges"].isnull().sum()
)

Missing TotalCharges: 0


Remove Customer ID

In [12]:
model_df = model_df.drop(
    columns=["customerID"]
)

print("Columns after removing customerID:")
print(model_df.columns.tolist())

Columns after removing customerID:
['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Churn']


Encode Churn

In [13]:
model_df["Churn"] = model_df["Churn"].map({
    "No": 0,
    "Yes": 1
})

print(model_df["Churn"].value_counts())

Churn
0    5174
1    1869
Name: count, dtype: int64


Separate Features and Target

In [14]:
X = model_df.drop(
    columns=["Churn"]
)

y = model_df["Churn"]

print("Features shape:", X.shape)
print("Target shape:", y.shape)

Features shape: (7043, 19)
Target shape: (7043,)


Identify Numerical and Categorical Columns

In [15]:
numeric_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object"]
).columns.tolist()

print("Numerical features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

Numerical features:
['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']

Categorical features:
['gender', 'Partner', 'Dependents', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']


Train/Test Split

In [16]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training data: (5634, 19)
Testing data: (1409, 19)


Check Churn Distribution

In [17]:
print("Training churn distribution:")
print(y_train.value_counts(normalize=True).round(3))

print("\nTesting churn distribution:")
print(y_test.value_counts(normalize=True).round(3))

Training churn distribution:
Churn
0    0.735
1    0.265
Name: proportion, dtype: float64

Testing churn distribution:
Churn
0    0.735
1    0.265
Name: proportion, dtype: float64


Create Preprocessing Pipeline

In [18]:
numeric_transformer = Pipeline(
    steps=[
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ]
)

Combine Preprocessor

In [19]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_transformer,
            numeric_features
        ),
        (
            "cat",
            categorical_transformer,
            categorical_features
        )
    ]
)

print("Preprocessing pipeline created successfully!")

Preprocessing pipeline created successfully!


Fit Preprocessor on Training Data

In [20]:
X_train_processed = preprocessor.fit_transform(
    X_train
)

X_test_processed = preprocessor.transform(
    X_test
)

print("Processed training shape:", X_train_processed.shape)
print("Processed testing shape:", X_test_processed.shape)

Processed training shape: (5634, 45)
Processed testing shape: (1409, 45)


Check Processed Data

In [21]:
print("Training data type:", type(X_train_processed))
print("Testing data type:", type(X_test_processed))

print(
    "Training missing values:",
    np.isnan(X_train_processed).sum()
)

print(
    "Testing missing values:",
    np.isnan(X_test_processed).sum()
)

Training data type: <class 'numpy.ndarray'>
Testing data type: <class 'numpy.ndarray'>
Training missing values: 0
Testing missing values: 0


Final Preprocessing Check

In [22]:
print("========== PREPROCESSING SUMMARY ==========")

print("Original rows:", len(model_df))
print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

print("Original features:", X.shape[1])
print("Processed training features:", X_train_processed.shape[1])
print("Processed testing features:", X_test_processed.shape[1])

print("Missing values after cleaning:")
print(model_df.isnull().sum().sum())

print("==========================================")

========== PREPROCESSING SUMMARY ==========
Original rows: 7043
Training rows: 5634
Testing rows: 1409
Original features: 19
Processed training features: 45
Processed testing features: 45
Missing values after cleaning:
0


# Phase 3 Summary

The customer churn dataset was prepared for machine learning by loading the data from the SQLite database and combining the four related tables.

The `TotalCharges` column was converted from object to numeric format. The 11 blank values identified during EDA were converted to missing values and replaced with zero because they belonged to customers with zero tenure.

The customer ID was removed because it is an identifier rather than a predictive feature. The `Churn` target variable was encoded as 0 for No and 1 for Yes.

The data was divided into 80% training data and 20% testing data using stratified sampling. Numerical features were standardized using `StandardScaler`, while categorical features were converted into numerical representations using `OneHotEncoder`.

The preprocessing transformations were fitted only on the training data and then applied to the test data. This helps prevent data leakage and prepares the dataset for machine learning model training.


In [23]:
# Save processed training and testing data

np.save("X_train_processed.npy", X_train_processed)
np.save("X_test_processed.npy", X_test_processed)
np.save("y_train.npy", y_train)
np.save("y_test.npy", y_test)

print("Processed data saved successfully!")

Processed data saved successfully!


In [24]:
import joblib

joblib.dump(preprocessor, "preprocessor.pkl")

print("Preprocessor saved successfully!")

Preprocessor saved successfully!
